# Notebook 04 — corrected windows and provisional HR/activity features

All 3,706 development EMAs and five frozen folds are retained. The 32 final-test participants remain sealed. This notebook displays aggregate reports and **synthetic examples only**; real participant rows stay in ignored Parquets.

The old int16 window grid overflowed from backward step 547. All three frozen IBI policies, both interval hypotheses and all lookbacks are reissued. The dependent Notebook 02 index/continuity report is reissued too. Old bundles are preserved but rejected as current inputs. Notebook 03's corrected pilot is unaffected. No expanded HRV, PSS, imputation, feature selection or model is calculated here.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402 — local source path
from garmin_hr_activity.features import prepare, window, CLOCK_FLAGS  # noqa: E402

PUBLIC = ROOT / "reports/hr_activity/7fdba29ad8d6f0fe"
verify_bundle(PUBLIC)

def read_aggregate(name):
    return json.loads((PUBLIC / f"{name}.json").read_text())

print("Verified aggregate report:", PUBLIC.relative_to(ROOT))

## Corrected time boundaries and supersession

The corrected audit has 5,321,816 windows; 3,294,634 clock locations change. Every row was independently checked using Python integer/datetime/timedelta arithmetic, including exact five-minute duration, one-minute stepping, both overflow transition points and the 24-hour boundary. All approved results through six hours are unchanged, including full eligibility diagnostics and coverage/attrition counts.

Every admitted interval is checked against the actual EMA cutoff. Sensor clocks are never shifted. EMA UTC is local start minus its row offset; minute-resolution starts get a 60-second buffer and second-resolution starts get none. Windows and direct feature lookbacks are left-inclusive/right-exclusive.

`reports/window_supersession.json` records the artifact graph and old/new/delta counts. Current loaders reject known-invalid versions even if directories are renamed. Historical checksum verification is a separate inspection operation, not authorization to use old windows.

In [ ]:
supersession = json.loads((ROOT / "reports/window_supersession.json").read_text())
changes = pd.DataFrame(supersession["coverage_delta"])
display(changes.loc[(changes.survey_category == "All") & (changes.lookback_minutes >= 720),
    ["policy", "scenario", "lookback_minutes", "obsolete_eligible_windows", "corrected_eligible_windows",
     "obsolete_eligible_ema", "corrected_eligible_ema", "corrected_eligible_participants"]])
print("Corrected eligibility:", supersession["corrected_eligibility"])
print("Corrected dependent index:", supersession["corrected_input_diagnostics"])

## One EMA → one feature row, with separate quality records

The ignored feature table has 3,706 rows and 104 measurement columns across all eight lookbacks. An explicit list includes 88 provisional predictors: nine HR summaries and supported steps/floor additions, each at eight lookbacks. Sixteen calorie/intensity placeholders remain missing and excluded. Identity, fold, category, cutoff, counts, gaps, freshness and missingness flags are not predictors.

HR accepts provisional 30–240 bpm readings without clock flags/conflicts. Mean, median, sample SD, min, max, Q25/Q75 require ≥2 distinct timestamps and ≥1 s span; latest needs one reading; OLS slope needs ≥3 readings and ≥60 s span, in bpm/minute. These describe observed readings, not every moment in the window. Quantile interpolation is an order-statistic definition, not sensor imputation.

Activity uses consecutive original timestamp groups. Both accepted endpoints must be fully within the window, same local day/offset/source, with 0<dt≤300 s and nondecreasing integer counters. Reject resets, conflicts, long gaps, rejected endpoints and boundary-crossing pairs. Never join across a rejected group. No supported pair means missing; an actual unchanged supported pair can produce zero. Sums are supported additions, not complete-window totals. Calorie units and intensity weighting/reset semantics remain unresolved.

In [ ]:
availability = pd.DataFrame(read_aggregate("availability"))
shown = availability[(availability.survey_category == "All") & (
    ((availability.sensor == "hr") & availability.metric.isin(["mean_bpm", "slope_bpm_per_minute"])) |
    availability.sensor.isin(["steps", "floorsClimbed"]))]
display(shown[["feature", "lookback_minutes", "ema_denominator", "available_ema", "available_participants", "no_recording_ema", "all_rejected_ema"]])
display(Image(filename=str(PUBLIC / "availability.png")))

## Survey-specific availability, gaps and overlap

Every category keeps its complete EMA denominator. HR/activity overlap means the same EMA has HR mean and at least one supported steps/floors feature, not necessarily simultaneous measurements. Larger lookbacks increase availability while leaving substantial gaps. Do not choose a lookback from retention alone or treat overlapping EMA windows as independent observations.

Quality is stored separately in 148,240 rows, one per EMA/sensor/lookback. Occupied minute bins describe recording density, not physiological duration. Supported counter time uses accepted original pair intervals; cohort time unions deduplicate repeated EMA views. No observed duration is manufactured by interpolation or multiplying window counts.

In [ ]:
display(Image(filename=str(PUBLIC / "survey_availability.png")))
display(Image(filename=str(PUBLIC / "overlap_and_gaps.png")))
gaps = pd.DataFrame(read_aggregate("gaps"))
display(gaps.loc[(gaps.survey_category == "All") & gaps.sensor.isin(["hr", "steps", "floorsClimbed"]),
    ["sensor", "lookback_minutes", "no_recording_ema", "all_rejected_ema", "median_reading_count",
     "median_latest_age_seconds", "median_maximum_internal_gap_seconds", "mean_occupied_minute_fraction"]])

## Synthetic row-level teaching example — no participant records

Treat synthetic seconds 0–300 as 11:55–12:00 for a second-resolution EMA starting at 12:00. HR [60,90,120] at seconds [0,60,120] yields mean/median 90, sample SD 30, Q25/Q75 75/105 and slope 30 bpm/min. The last reading is 180 seconds old; freshness stays outside the predictor list. A reading at second 300 is excluded.

For a counter [100,110,3,8], the reset from 110 to 3 contributes nothing. Valid additions are 10+5=15. Moving the left boundary beyond the first reading excludes its crossing increment entirely. These examples explain the fixed rules; they are not used to tune retention.

In [ ]:
config = json.loads((ROOT / "config/hr_activity.json").read_text())

def synthetic_source(times, values):
    frame = pd.DataFrame({"n_epoch_s": np.array(times, float), "n_value": np.array(values, float),
        "n_logical_row": np.arange(len(times)), "n_text_local": pd.to_datetime(np.array(times) + 43200, unit="s"),
        "n_inferred_offset_minutes": 0, "n_source_sha256": "SYNTHETIC_SOURCE", "q_conflicting_timestamp_value": False})
    for flag in CLOCK_FLAGS:
        frame[flag] = False
    return frame

grouped, _ = prepare(synthetic_source([0, 60, 120, 300], [60, 90, 120, 240]), "hr", config)
values, quality, _ = window(grouped, 0, 300, "hr", config)
display(pd.DataFrame([{"example": "SYNTHETIC EMA feature row", **values}]))
display(pd.DataFrame([{"example": "SYNTHETIC separate quality", "readings": quality["valid_readings"], "freshness_seconds": quality["latest_age_seconds"]}]))
assert values["mean_bpm"] == 90 and values["sd_bpm"] == 30 and values["slope_bpm_per_minute"] == 30
counter, _ = prepare(synthetic_source([0, 10, 20, 30], [100, 110, 3, 8]), "steps", config)
full, diagnostics, _ = window(counter, 0, 40, "steps", config)
clipped, _, _ = window(counter, 5, 40, "steps", config)
assert full["added_supported"] == 15 and clipped["added_supported"] == 5
print("SYNTHETIC supported steps:", full["added_supported"], "; reset excluded; moving left boundary gives", clipped["added_supported"])

## Limitations, provider questions and reproduction

Timing remains provisional: acquisition/receipt, batching and counter observation intervals are unresolved. HR averages are reading-weighted; counter additions cover supported pairs only. No lookback or threshold has been selected from retention. The provider-question draft asks about timestamp meaning, IBI anchoring/missed beats, device/quality annotations, and activity units/reset periods. **Draft only; not sent.**

Read `reports/hr_activity/7fdba29ad8d6f0fe/report.md`, its `feature_dictionary.json`, `reports/hr_activity_feature_specification.md`, `reports/window_supersession.json` and `reports/provider_questions_draft.md`.

Private feature and quality datasets are under `private/hr_activity/features/76f5885b3955d5b0/`. Reusable commands require the configured reference location and reuse immutable normalized data:

```sh
PYTHONPATH=src uv run python -m garmin_hr_activity.reissue
PYTHONPATH=src uv run python -m garmin_hrv_input.diagnostics
PYTHONPATH=src uv run python -m garmin_hr_activity.pipeline
```

Existing validated versions are hash-checked and reused, never overwritten. Source notebook outputs stay empty; execute copies into ignored `outputs/`. No expanded HRV, Garmin stress predictors, PSS targets, imputation, models, commit or push.